# 08 -- TAQ Millisecond Intraday Indicators Data Collection

## Purpose
Downloads WRDS Intraday Indicators (IID) from the TAQ millisecond database for every stock in the top-100 S&P 500 universe, providing ~205 daily microstructure factors per stock.

## Source
WRDS TAQ millisecond database (`taqmsec.wrds_iid_YYYY` tables, one per year) and the TAQ-CRSP linking table (`wrdsapps.taqmclink`), via the `wrds` Python library authenticated with username `henrylavender`.

## Input
`Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet` -- master PERMNO list from notebook 05.

## Identifier Linking
TAQ uses ticker symbols (`sym_root`) rather than PERMNOs. The mapping is handled in two layers:

**Automated linking:** The `wrdsapps.taqmclink` table maps `(sym_root, date)` to `permno`. The table is filtered to the master PERMNO list, deduplicated to keep only the best `match_lvl` per `(sym_root, date)`, and saved for reference.

**Manual overrides for 13 PERMNOs:** Some stocks are not matched by the automated link due to ticker changes, multi-class share structures, or suffix conventions. Manual mappings were identified by checking which master PERMNOs returned zero rows after the initial automated pull:
- `LYB` (LyondellBasell), `KMI` (Kinder Morgan), `SLB` (Schlumberger), `LIN` (Linde), `JCI` (Johnson Controls), `CB` (Chubb), `RIG` (Transocean), `SPG` (Simon Property Group), `PLD` (Prologis), `ACN` (Accenture), `COV` (Covidien)
- `BRK` with `sym_suffix = 'B'` (Berkshire Hathaway class B)
- `CMCS` with `sym_suffix = 'A'` (Comcast, stored as CMCS.A in TAQ)
- Historical ticker: `ACE` for Chubb (pre-2016 rename)

Manual overrides take priority over automated matches when both exist.

## Collection Method
Data is downloaded year by year (2004--2024) from the corresponding `taqmsec.wrds_iid_YYYY` table. Each query filters to the combined set of automated and manual `sym_root` values, restricted to primary share class (`sym_suffix IS NULL` or empty string) plus the specific suffix exceptions for BRK.B and CMCS.A.

For each year:
1. Raw TAQ rows are retrieved for all relevant sym_roots
2. PERMNOs are assigned via manual lookup first, then automated link table match as fallback
3. Rows with no PERMNO match are dropped
4. Deduplicated to one row per `(permno, date)`, keeping the first occurrence

## Variables Collected
All columns from the `wrds_iid_YYYY` tables are retained (~205 microstructure factors). These include measures of spreads, volatility, volume, order flow, trade classification, and related intraday statistics. The `sym_root` and `sym_suffix` columns are dropped after PERMNO assignment. A `year` column is added for parquet partitioning.

## Outputs
Saved to `Data/Data_Collection/Initial/08_TAQ_Millisecond/`:
- `taq_link.parquet` -- filtered and deduplicated TAQ-CRSP linking table for reference
- `firm_daily_taq/` -- partitioned parquet by year (subfolders `year=2004/`, `year=2005/`, etc.)

In [ ]:
# %% [markdown]
# # Stage 4: Collect TAQ Millisecond Intraday Indicators (IID)
#
# Downloads WRDS Intraday Indicators from the TAQ millisecond database for every
# stock in our top-100 S&P 500 universe.
#
# TAQ uses ticker symbols (`sym_root`) rather than PERMNOs. We use the TAQ-CRSP
# linking table (`wrdsapps.taqmclink`) to map sym_root → permno, with manual
# overrides for ~13 tickers where the automated link fails (ticker changes,
# multi-class stocks like BRK.B, suffix issues like CMCS.A → CMCSA).
#
# Outputs (saved to ../../Data/Data_Collection/Initial/08_TAQ_Millisecond/):
#   taq_link.parquet      — filtered TAQ-CRSP linking table for reference
#   firm_daily_taq/       — partitioned parquet by year with ~205 microstructure factors

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import gc
import time

OUTPUT_DIR = Path('../../Data/Data_Collection/Initial/08_TAQ_Millisecond')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conn = wrds.Connection(wrds_username='henrylavender')

START_YEAR = 2004
END_YEAR = 2024

# %% [markdown]
# ## Step 1: Load master PERMNO list and build the TAQ-CRSP linking table

# %%
master = pd.read_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet')
permno_list = master['permno'].tolist()
permno_str = ','.join(str(p) for p in permno_list)

print(f"Master list: {len(permno_list)} PERMNOs")
print("Downloading TAQ-CRSP linking table...")

link = conn.raw_sql(f"""
    SELECT date, sym_root, sym_suffix, permno, match_lvl
    FROM wrdsapps.taqmclink
    WHERE permno IN ({permno_str})
""", date_cols=['date'])

print(f"Raw link table: {len(link):,} rows, {link['permno'].nunique()} PERMNOs, "
      f"{link['sym_root'].nunique()} unique sym_roots")

# Keep only the best match level per (sym_root, date)
link = link.sort_values('match_lvl').drop_duplicates(
    subset=['sym_root', 'date'], keep='first'
)

# Save for reference
link.to_parquet(OUTPUT_DIR / 'taq_link.parquet', index=False, engine='pyarrow')
print(f"After dedup: {len(link):,} rows")
print(f"Match level distribution:\n{link['match_lvl'].value_counts().sort_index().to_string()}")

# %% [markdown]
# ## Step 2: Identify PERMNOs that the automated link misses
#
# Some tickers aren't matched by taqmclink due to ticker changes, multi-class
# stocks (BRK.B stored as sym_root='BRK' sym_suffix='B'), or suffix conventions
# (CMCSA stored as sym_root='CMCS' sym_suffix='A'). We handle these with manual
# overrides so everything is collected in a single pass.

# %%
# ── Manual overrides for PERMNOs that taqmclink misses ──────────────────────
# Format: permno → list of sym_roots that TAQ uses for this stock.
# These were identified by checking which master PERMNOs had zero rows after
# the initial automated pull.

MANUAL_FIXES = {
    12345: ('LYB',    None),    # LyondellBasell
    12558: ('KMI',    None),    # Kinder Morgan
    14277: ('SLB',    None),    # Schlumberger
    18143: ('LIN',    None),    # Linde
    45356: ('JCI',    None),    # Johnson Controls
    79057: ('CB',     None),    # Chubb (was ACE before rename)
    79237: ('RIG',    None),    # Transocean
    80100: ('SPG',    None),    # Simon Property Group
    83443: ('BRK',    'B'),     # Berkshire Hathaway class B
    85592: ('PLD',    None),    # Prologis
    89071: ('ACN',    None),    # Accenture
    89525: ('CMCS',   'A'),     # Comcast — stored as CMCS.A in TAQ
    92156: ('COV',    None),    # Covidien
}

# Also include historical tickers for stocks that changed names
MANUAL_EXTRA_ROOTS = {
    79057: [('ACE', None)],     # Chubb was ACE before 2016
}

# Build the complete set of (sym_root, sym_suffix, permno) for manual queries
manual_queries = []
for permno, (root, suffix) in MANUAL_FIXES.items():
    manual_queries.append((root, suffix, permno))

for permno, extras in MANUAL_EXTRA_ROOTS.items():
    for root, suffix in extras:
        manual_queries.append((root, suffix, permno))

manual_sym_roots = set(q[0] for q in manual_queries)

print(f"Manual overrides: {len(MANUAL_FIXES)} PERMNOs, "
      f"{len(manual_queries)} (sym_root, suffix) pairs")
print(f"Manual sym_roots: {sorted(manual_sym_roots)}")

# %% [markdown]
# ## Step 3: Build query components

# %%
# ── Automated sym_roots from the link table ──────────────────────────────────
# Deduplicate: one sym_root per (permno, date), prefer best match_lvl
link_for_join = (
    link[['sym_root', 'date', 'permno', 'match_lvl']]
    .sort_values('match_lvl')
    .drop_duplicates(subset=['permno', 'date'], keep='first')
    [['sym_root', 'date', 'permno']]
)

auto_sym_roots = sorted(link['sym_root'].unique().tolist())
print(f"Automated sym_roots: {len(auto_sym_roots)}")

# ── Combined sym_root list for SQL WHERE clause ─────────────────────────────
all_sym_roots = sorted(set(auto_sym_roots) | manual_sym_roots)
sym_root_str = ','.join(f"'{s}'" for s in all_sym_roots)
print(f"Total sym_roots to query: {len(all_sym_roots)}")

# ── Check which TAQ IID tables exist ────────────────────────────────────────
available_tables = conn.list_tables(library='taqmsec')
iid_tables = sorted([t for t in available_tables if t.startswith('wrds_iid_')])
print(f"Available IID tables: {len(iid_tables)} ({iid_tables[0]} to {iid_tables[-1]})")

# %% [markdown]
# ## Step 4: Download TAQ IID data year by year
#
# For each year:
# 1. Query TAQ with all sym_roots (automated + manual), filtering to primary
#    share class (sym_suffix IS NULL or '') plus specific suffixes for manual
#    overrides (BRK.B, CMCS.A).
# 2. Match rows to PERMNOs via the link table (automated) or manual mapping.
# 3. Deduplicate to one row per (permno, date).

# %%
all_years = []
total_rows = 0

for year in range(START_YEAR, END_YEAR + 1):
    table_name = f'wrds_iid_{year}'

    if table_name not in iid_tables:
        print(f"  {year}: TABLE NOT FOUND — skipping")
        continue

    t0 = time.time()

    # Query: primary share class (NULL/empty suffix) for all stocks,
    # plus ticker-specific suffix exceptions for BRK.B and CMCS.A only
    query = f"""
        SELECT t.*
        FROM taqmsec.{table_name} t
        WHERE t.sym_root IN ({sym_root_str})
          AND (
              t.sym_suffix IS NULL
              OR t.sym_suffix = ''
              OR (t.sym_root = 'BRK' AND t.sym_suffix = 'B')
              OR (t.sym_root = 'CMCS' AND t.sym_suffix = 'A')
          )
    """

    df_year = conn.raw_sql(query, date_cols=['date'])
    n_raw = len(df_year)

    if df_year.empty:
        print(f"  {year}: 0 rows returned — skipping")
        continue

    # ── Assign PERMNOs ──────────────────────────────────────────────────────

    # Step A: Manual overrides (applied first so they take priority)
    # Build a lookup: (sym_root, sym_suffix) → permno
    manual_lookup = {}
    for root, suffix, permno in manual_queries:
        key_suffix = suffix if suffix is not None else ''
        manual_lookup[(root, key_suffix)] = permno

    df_year['_suffix_clean'] = df_year['sym_suffix'].fillna('').astype(str).str.strip()
    df_year['_manual_key'] = list(zip(df_year['sym_root'], df_year['_suffix_clean']))
    df_year['permno_manual'] = df_year['_manual_key'].map(manual_lookup)

    # Step B: Automated link table match
    link_year = link_for_join[
        (link_for_join['date'] >= pd.Timestamp(f'{year}-01-01')) &
        (link_for_join['date'] <= pd.Timestamp(f'{year}-12-31'))
    ]

    df_year = df_year.merge(
        link_year[['sym_root', 'date', 'permno']].rename(columns={'permno': 'permno_auto'}),
        on=['sym_root', 'date'],
        how='left'
    )

    # Step C: Combine — manual takes priority, fall back to automated
    df_year['permno'] = df_year['permno_manual'].fillna(df_year['permno_auto'])

    # Drop rows with no PERMNO match at all
    n_unmatched = df_year['permno'].isna().sum()
    df_year = df_year.dropna(subset=['permno'])
    df_year['permno'] = df_year['permno'].astype(int)

    # Drop helper columns
    df_year = df_year.drop(
        columns=['sym_root', 'sym_suffix', '_suffix_clean', '_manual_key',
                 'permno_manual', 'permno_auto'],
        errors='ignore'
    )

    # Deduplicate: one row per (permno, date)
    before = len(df_year)
    df_year = df_year.drop_duplicates(subset=['permno', 'date'], keep='first')
    n_dupes = before - len(df_year)

    df_year['year'] = year

    elapsed = time.time() - t0
    n_permnos = df_year['permno'].nunique()

    print(f"  {year}: {n_raw:,} raw → {len(df_year):,} matched "
          f"({n_unmatched:,} unmatched, {n_dupes:,} dupes) | "
          f"{n_permnos} PERMNOs | {elapsed:.1f}s")

    total_rows += len(df_year)
    all_years.append(df_year)
    del df_year, link_year
    gc.collect()

print(f"\nTotal: {total_rows:,} rows across {len(all_years)} years")

# %% [markdown]
# ## Step 5: Concatenate and save

# %%
print("Concatenating all years...")
df = pd.concat(all_years, ignore_index=True)
del all_years
gc.collect()

df = df.sort_values(['permno', 'date']).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# %%
out_path = OUTPUT_DIR / 'firm_daily_taq'
if out_path.exists():
    shutil.rmtree(out_path)

df.to_parquet(out_path, partition_cols=['year'], engine='pyarrow', index=False)

year_folders = sorted(out_path.glob('year=*'))
print(f"Saved to {out_path} with {len(year_folders)} year partitions")

# %% [markdown]
# ## Step 6: Verification

# %% [markdown]
# ### Basic dimensions

# %%
print(f"Total rows: {len(df):,}")
print(f"Unique PERMNOs: {df['permno'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Columns: {len(df.columns)}")

# %% [markdown]
# ### Rows and PERMNOs per year

# %%
yearly = df.groupby('year').agg(rows=('permno', 'size'), permnos=('permno', 'nunique'))
print("Rows and PERMNOs per year:")
print(yearly.to_string())
print(f"\nMean rows/year: {yearly['rows'].mean():,.0f}")

# %% [markdown]
# ### Coverage check: which master PERMNOs are missing?

# %%
taq_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'].tolist())
missing = master_permnos - taq_permnos

print(f"TAQ coverage: {len(taq_permnos)}/{len(master_permnos)} master PERMNOs "
      f"({len(taq_permnos)/len(master_permnos)*100:.1f}%)")
if missing:
    print(f"Missing PERMNOs ({len(missing)}): {sorted(missing)}")
else:
    print("All master PERMNOs matched in TAQ.")

# %% [markdown]
# ### Null % for key TAQ columns

# %%
key_cols = [
    'effectivespread_percent_ave', 'quotedspread_percent_tw',
    'ivol_t', 'ivol_q', 'buyvol_lr', 'sellvol_lr',
    'total_vol_m', 'total_n_trades_m',
    'buyvol_retail', 'sellvol_retail',
    'buyvol_inst20k', 'sellvol_inst20k',
    'tsignsqrtdvol1', 'bs_ratio_num'
]

df_cols_lower = {c.lower(): c for c in df.columns}
key_existing = [df_cols_lower[k] for k in key_cols if k in df_cols_lower]

if key_existing:
    null_pct = (df[key_existing].isnull().sum() / len(df) * 100).round(1)
    print("Null % for key TAQ columns:")
    for col, pct in null_pct.items():
        print(f"  {col:45s} {pct:5.1f}%")

# %% [markdown]
# ### Sample data for AAPL in 2020

# %%
sample_permno = 14593
sample_cols_wanted = ['permno', 'date', 'effectivespread_percent_ave',
                      'ivol_t', 'buyvol_lr', 'sellvol_lr', 'total_vol_m']
sample_cols = [c for c in sample_cols_wanted if c in df.columns]

sample = df[(df['permno'] == sample_permno) & (df['year'] == 2020)]
if sample.empty:
    sample_permno = df['permno'].iloc[0]
    sample = df[(df['permno'] == sample_permno) & (df['year'] == 2020)]
    print(f"AAPL not found; showing PERMNO {sample_permno} instead")

print(f"Sample: PERMNO {sample_permno} in 2020 (first 5 rows):")
print(sample[sample_cols].head(5).to_string(index=False))

# %% [markdown]
# ## Cleanup

# %%
conn.close()
del df
gc.collect()

print(f"\nStage 4 complete. Files saved to {OUTPUT_DIR}:")
print(f"  taq_link.parquet")
print(f"  firm_daily_taq/ (partitioned parquet by year)")